# Exercise 03 Workbook: simulate your own protein

This notebook is **deliberately almost empty**, and it is entirely about **your** protein:
the one you picked in ex01's Character Sheet and predicted in ex02.

Its companion, **`ex03_guide.ipynb`**, works one simulation through in full:
2JAC (yeast glutaredoxin-1) in explicit solvent, from raw PDB download to per-residue
flexibility. **Read the guide first.** Then work here, with it
open beside you.

## How to work in this notebook

- **Copy code across from the guide and adapt it.** The setup, minimisation, equilibration
  and MDAnalysis cells are all generic. Reusing them is the point.
- **Structure the notebook however you like.** The sections below are a checklist of what
  has to be here, not a cell-by-cell template.
- **The reasoning is what's graded, not the code.** Every section asks for a written
  conclusion as well as output.
- **Start early.** A simulation that has to reach a stable RMSD is the one part of this
  course you cannot rush the night before.

**Time: about 60 minutes of your work**, plus the runs themselves. **What you will not learn
here** is anything the guide did not show: each section repeats a guide section on your protein.

## Using AI Tools

You may use AI assistants for syntax, debugging, and code snippets. You must supply the
physical reasoning, the justification for your simulation choices, and the critical
evaluation of what came back. **"The AI said so" is not an answer.**

**Predict-first prompts** (sections 2 and 5b) ask for your prediction *before* you run
anything. That written prediction is the graded artefact.

## Setup

Copy the guide's install and imports cells here and run them first. A Colab notebook
starts empty, so nothing from the guide's session carries over here. The install takes up
to a minute or two.

In [2]:
#@title Install the packages {display-mode: "form"}
if "google.colab" in str(get_ipython()):
    print("Running on colab")
    print("Installing required packages...")
    !pip install -q openmm==8.2.0 mdanalysis==2.10.0 py3dmol==2.4.0 rdkit==2026.3.5
    %pip install -q pdbfixer==1.12.0
    print("Installation complete!")
else:
    print("Not running on colab.")
    print("Make sure you have all required packages installed in your environment!")

Running on colab
Installing required packages...
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Installation complete!


In [3]:
#@title Imports {display-mode: "form"}
import os
import subprocess
import sys
import tempfile
import urllib.request
from pathlib import Path
from sys import stdout

import matplotlib.pyplot as plt
import pandas as pd
import py3Dmol
import MDAnalysis as mda
from MDAnalysis.analysis import rms, align
from rdkit import Chem
from rdkit.Chem import rdMolAlign

from pdbfixer import PDBFixer
from openmm.app import (
    CheckpointReporter,
    DCDReporter,
    ForceField,
    HBonds,
    Modeller,
    PDBFile,
    PME,
    Simulation,
    StateDataReporter,
)
from openmm import LangevinMiddleIntegrator, MonteCarloBarostat, XmlSerializer
from openmm.unit import (
    Quantity,
    amu,
    bar,
    kelvin,
    nanometer,
    picosecond,
    picoseconds,
    seconds,
)

print("Libraries loaded")

Libraries loaded


Then copy the guide's cell that builds the second environment for the ligand's
parameters. It takes about a minute and does not restart the kernel.

In [4]:
#@title Build the ligand's parametrisation environment {display-mode: "form"}
# The ligand's parameters are built in their own environment, not in this kernel. The
# versions are pinned: the charge model and the force field below are only valid as a pair.
OPENFF_STACK = ("python=3.13 openff-toolkit=0.19.0 openmmforcefields=0.16.0 "
                "openff-forcefields=2026.09.0 openff-nagl=0.6.1 openff-nagl-models=2026.09.0 "
                "rdkit=2026.03.1 openmm=8.6.1")
# Dimorphite-DL protonates the ligand for pH 7. It declares rdkit<2026, but gives the same
# states on RDKit 2026, so pip adds it without touching the RDKit pinned above.
PROTONATION = "dimorphite_dl==2.0.2 loguru==0.7.3"

if "google.colab" in str(get_ipython()):
    PARAM_ENV = Path("/opt/openff-env")
    if not (PARAM_ENV / "bin" / "python").exists():
        print("Installing micromamba + the OpenFF stack (about a minute, once per session)...")
        !curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /usr/local bin/micromamba
        !micromamba create -y -q -p {PARAM_ENV} -c conda-forge {OPENFF_STACK}
        !{PARAM_ENV}/bin/python -m pip install -q --no-deps {PROTONATION}
    PARAM_PYTHON = str(PARAM_ENV / "bin" / "python")
else:
    print("Not running on colab.")
    PARAM_ENV = Path.home() / "openff-env"
    print("The ligand needs its own environment. Create it once, outside this notebook:")
    print(f"    micromamba create -p {PARAM_ENV} -c conda-forge {OPENFF_STACK}")
    print(f"    {PARAM_ENV}/bin/python -m pip install --no-deps {PROTONATION}")
    PARAM_PYTHON = (str(PARAM_ENV / "bin" / "python")
                    if (PARAM_ENV / "bin" / "python").exists() else sys.executable)

# Fail here, loudly, rather than three cells later.
_check = subprocess.run(
    [PARAM_PYTHON, "-c", "import dimorphite_dl, openff.toolkit as t; print(t.__version__)"],
    capture_output=True, text=True)
if _check.returncode == 0:
    print(f"Parametrisation interpreter ready: {PARAM_PYTHON} "
          f"(openff-toolkit {_check.stdout.strip()})")
else:
    print("The OpenFF stack is NOT available; the ligand cells below will fail:")
    print(_check.stderr[-500:])

Installing micromamba + the OpenFF stack (about a minute, once per session)...
Parametrisation interpreter ready: /opt/openff-env/bin/python (openff-toolkit 0.19.0)


In [5]:
# helpers
def fetch_pdb(pdb_id: str, path: str | None = None) -> str:
    """Download one PDB entry next to the notebook, once. Returns the local path.

    Fetched with Python rather than wget, so this works identically on Colab, in a local
    conda environment, and in a container: the download is not a Colab-only step.
    MDAnalysis reads structures but does not retrieve them, so this stays plain urllib.
    """
    path = path or f"{pdb_id}.pdb"
    if not os.path.exists(path):
        urllib.request.urlretrieve(f"https://files.rcsb.org/download/{pdb_id}.pdb", path)
        print(f"Downloaded {path} from RCSB")
    else:
        print(f"{path} already present")
    return path


def prepare_structure(pdb_path: str, out_path: str, keep_water: bool = False,
                      ph: float = 7.0) -> str:
    """Run PDBFixer over a raw PDB entry and write a protein OpenMM can build.

    Prints what was missing before fixing it: a fixer that silently added twelve residues
    has made a modelling decision on your behalf, and you are responsible for it either
    way. Heteroatoms are removed here; the ligand is handled separately, because it needs
    parameters this force field does not have. Modified amino acids, such as the
    selenomethionine many crystals carry, become the standard one first: left alone,
    `removeHeterogens` would cut them out of the chain. Residue numbers stay the ones the
    input file uses.
    """
    fixer = PDBFixer(filename=pdb_path)
    fixer.findMissingResidues()
    # One entry per gap, (chain, position in chain) -> the residue names to insert there.
    chains = list(fixer.topology.chains())
    gaps = []
    for (chain, position), names in fixer.missingResidues.items():
        present = list(chains[chain].residues())
        where = (f"after residue {present[position - 1].id}" if position
                 else f"before residue {present[0].id}")
        gaps.append(f"chain {chains[chain].id}, {where}: {' '.join(names)}")
    fixer.findNonstandardResidues()
    nonstandard = [f"{residue.name}{residue.id}" for residue, _ in fixer.nonstandardResidues]
    fixer.replaceNonstandardResidues()
    fixer.removeHeterogens(keepWater=keep_water)
    fixer.findMissingAtoms()

    n_missing = sum(len(names) for names in fixer.missingResidues.values())
    print(f"Missing residues found: {n_missing} in {len(gaps)} gap(s)")
    for gap in gaps:
        print(f"    {gap}")
    print(f"Missing atoms found   : {len(fixer.missingAtoms)}")
    print(f"Nonstandard residues  : {nonstandard}")

    fixer.addMissingAtoms()
    fixer.addMissingHydrogens(ph)
    with open(out_path, "w") as out:
        PDBFile.writeFile(fixer.topology, fixer.positions, out, keepIds=True)

    print(f"\nPrepared {out_path}: {fixer.topology.getNumAtoms()} atoms, "
          f"{fixer.topology.getNumResidues()} residues")
    return out_path


def ligand_from_crystal(pdb_path: str, code: str, out_path: str | None = None) -> str:
    """One copy of a named species, in one conformation, as its own PDB file.

    Both filters earn their place. A crystal often models the ligand more than once, so we
    take the first residue. And where an atom was modelled in two **alternate positions**,
    MDAnalysis returns both as one residue: parametrising those overlapping copies as
    though they were a single molecule is a silent way to get nonsense. Entries with no
    alternates leave the field blank, which is why the blank is kept alongside "A".
    """
    universe = mda.Universe(pdb_path)
    ligand = universe.select_atoms(f"resname {code}")
    if not len(ligand):
        raise ValueError(f"no {code} residues in {pdb_path}")

    ligand = ligand.residues[0].atoms.select_atoms("altloc A or not altloc ?")

    out_path = out_path or f"{code}.pdb"
    ligand.write(out_path)
    print(f"Wrote {out_path}: {len(ligand)} atoms of {code}")
    return out_path


def parametrise_ligand(ligand_pdb: str, code: str,
                       python: str | None = None) -> tuple[str, str]:
    """Parameters for one ligand, built in the OpenFF environment.

    Runs as a separate process because that toolchain cannot be imported from this
    kernel. Returns (parameter file, ligand with hydrogens), and both are ordinary files:
    ForceField reads the first exactly as it reads Amber's own.
    """
    out_xml, out_pdb = f"{code}.xml", f"{code}_h.pdb"

    # A notebook kernel exports settings that only make sense inside it. MPLBACKEND
    # names Jupyter's inline backend, which the other environment does not have, and
    # PYTHONPATH points at this kernel's packages. Inherited, either one breaks the
    # subprocess with an error that looks nothing like its cause.
    environment = {key: value for key, value in os.environ.items()
                   if key not in ("MPLBACKEND", "PYTHONPATH", "PYTHONHOME")}

    result = subprocess.run(
        [python or PARAM_PYTHON, "_parametrise_ligand.py",
         str(ligand_pdb), code, out_xml, out_pdb],
        capture_output=True, text=True, env=environment)
    print(result.stdout.strip())
    if result.returncode != 0:
        print(result.stderr[-1500:])
        raise RuntimeError(f"parametrising {code} failed")
    return out_xml, out_pdb


---

## Your structure: the crystal from ex01

This notebook works on **the crystal you chose in ex01**, the PDB entry on your Character
Sheet, and on no other: not another entry of the same protein, even a sharper one. The
residue numbers you carry from one exercise to the next belong to that entry. Every
candidate numbers its residues the way UniProt does, so they also line up with ex01's conserved positions and ex02's AlphaFold model.

Put its PDB ID at the top of your first code cell, as `PDB_ID = "..."`, and use `PDB_ID`
wherever the guide names its own entry.

In [6]:
PDB_ID = "1A9T"
CHAIN  = "A"
LIGAND = "R1P"

---

## 1. Prepare your protein and its ligand

Fix your structure (`PDB_ID`) with PDBFixer and give your ligand parameters of its own, the
way the guide's section 1 does.

**Report:** what was wrong with your structure, what the parametrisation printed for your
ligand, and the **net charge** it assigned for pH 7. Say whether you agree with it, and why:
a group buried in the pocket or next to a metal can have its pKa shifted.

In [7]:
pdb_path = fetch_pdb(pdb_id=PDB_ID)


Downloaded 1A9T.pdb from RCSB


In [8]:
fixed_path = prepare_structure(pdb_path=pdb_path, out_path="1A9T_fixed.pdb")

Missing residues found: 0 in 0 gap(s)
Missing atoms found   : 0
Nonstandard residues  : []

Prepared 1A9T_fixed.pdb: 4391 atoms, 284 residues


In [9]:
%%writefile _parametrise_ligand.py
# Runs in the OpenFF environment, not in this notebook's kernel.
# Usage: python _parametrise_ligand.py <ligand.pdb> <CODE> <out.xml> <out.pdb>
import json
import sys
import urllib.request

from dimorphite_dl import protonate_smiles
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem
from openff.toolkit import Molecule
from openmm.app import PDBFile, Topology
from openmm.app.element import Element
from openmmforcefields.generators import SMIRNOFFTemplateGenerator

RDLogger.DisableLog("rdApp.*")
ligand_pdb, code, out_xml, out_pdb = sys.argv[1:5]

# 1. the chemistry a PDB file does not carry
record = json.load(urllib.request.urlopen(
    f"https://data.rcsb.org/rest/v1/core/chemcomp/{code}", timeout=30))
smiles = next(d["descriptor"] for d in record["pdbx_chem_comp_descriptor"]
              if d.get("type") == "SMILES_CANONICAL"
              and d.get("program") == "OpenEye OEToolkits")

# 2. the protonation state at pH 7, from typical pKa values. A phosphorus-acid proton the
# table has no rule for (a phosphorothioate) goes too: those acids have pKa below 3.
template = Chem.MolFromSmiles(
    protonate_smiles(smiles, ph_min=7.0, ph_max=7.0, precision=0.0)[0])
for _, acid in template.GetSubstructMatches(
        Chem.MolFromSmarts("[PX4;$(P=[O,S])][OX2H1,SX2H1]")):
    template.GetAtomWithIdx(acid).SetFormalCharge(-1)
    template.GetAtomWithIdx(acid).SetNumExplicitHs(0)
Chem.SanitizeMol(template)

# removeHs is load-bearing: some entries deposit explicit hydrogens, and the dictionary
# template is heavy-atom only, so matching with hydrogens present fails on valence.
crystal = Chem.MolFromPDBBlock(open(ligand_pdb).read(), removeHs=True)
mol = Chem.AddHs(AllChem.AssignBondOrdersFromTemplate(template, crystal), addCoords=True)

# 3. partial charges, from a graph neural network rather than from quantum chemistry
ligand = Molecule.from_rdkit(mol, allow_undefined_stereo=True)
ligand.name = code
ligand.assign_partial_charges("openff-gnn-am1bcc-1.0.0.pt")

# 4. the residue template, as an ordinary OpenMM parameter file. Sage 2.3.0 is named, not
# "the newest installed": its parameters were refit to exactly the charges above, and the
# two are only validated as a pair.
forcefield = "openff-2.3.0"
generator = SMIRNOFFTemplateGenerator(molecules=ligand, forcefield=forcefield)
open(out_xml, "w").write(generator.generate_residue_template(ligand))

# The coordinates go out afterwards, and the topology is built by hand rather than
# converted. Converting splits the hydrogens into a chain of their own, and OpenMM then
# reads them back as a second residue holding nothing but hydrogens, which matches no
# template. One chain, one residue, every atom and every bond.
topology = Topology()
residue = topology.addResidue(code, topology.addChain())
counts, atoms = {}, []
for atom in ligand.atoms:
    element = Element.getByAtomicNumber(atom.atomic_number)
    counts[element.symbol] = counts.get(element.symbol, 0) + 1
    atoms.append(topology.addAtom(f"{element.symbol}{counts[element.symbol]}",
                                  element, residue))
for bond in ligand.bonds:
    topology.addBond(atoms[bond.atom1_index], atoms[bond.atom2_index])

with open(out_pdb, "w") as handle:
    PDBFile.writeFile(topology, ligand.conformers[0].to_openmm(), handle)

print(f"{code}: {mol.GetNumAtoms()} atoms with hydrogens, formal charge "
      f"{Chem.GetFormalCharge(mol):+d}, {forcefield} -> {out_xml}")


Writing _parametrise_ligand.py


In [10]:
# What else is in this crystal? MDAnalysis reads a structure file as happily as it reads a
# trajectory, and the selection language is the same one we use on the trajectory later.
crystal = mda.Universe(pdb_path)
other = crystal.select_atoms("not protein and not resname HOH")
print(f"Non-protein, non-water species in {pdb_path}: "
      f"{sorted(set(other.residues.resnames))}")

ligand_crystal = ligand_from_crystal(pdb_path=pdb_path, code=LIGAND)
ligand_xml, ligand_pdb = parametrise_ligand(ligand_pdb=ligand_crystal, code=LIGAND)


Non-protein, non-water species in 1A9T.pdb: ['HPA', 'R1P']
Wrote R1P.pdb: 14 atoms of R1P


/usr/local/lib/python3.13/dist-packages/MDAnalysis/coordinates/PDB.py:1282: UserWarning: Found no information for attr: 'formalcharges' Using default value of '0'
  warnings.warn(


R1P: 23 atoms with hydrogens, formal charge -2, openff-2.3.0 -> R1P.xml


## 2. Predict first, before you simulate

**Before running anything:** which region of your protein will be **most flexible**, and which
most rigid? Will your ligand's pocket move more than your termini, and where will the
**residues gripping it** sit on that scale? Name residues; base it on your **ex01**
B-factors and unresolved residues and your **ex02** pLDDT. If those two disagree, say so.

> **My prediction (most flexible, most rigid):**
> both ends will be most flexible, resiudes 33-36, 56-69, and 248-256 will also be flixible, but not as flexible as the ends. Binding site and the core will be most rigid.
>
> **My prediction (pocket, gripping residues):**
> the binding site itself will be rigid, but the ends of the binding site will be mobile.
>
> **My reasoning, citing ex01 and ex02:**
> Both findings from ex01 and ex02 suggest those predictions. As the B-factors indicate that  residues 1–4 and 279–284, so the ends, are highly felxible -->  B-factors of
  67.9 Å² at `VAL284` and 67.1 Å² at `MET1`, against a mean of 13.6 Å². Write here something fom the pLDDT plot from ex02

## 3. Run the simulation

Follow the guide's section 3, and state and justify your choices: **solvent** (explicit water,
as the guide does, or vacuum, a different physical system), **force field**, **integrator**,
**time step**, **temperature**, and **how long** you ran. Run until RMSD and potential energy
stabilise, and say what you would do with more time.

In [11]:
# Load the PREPARED protein -- never the raw download -- and the parametrised ligand.
pdb = PDBFile(fixed_path)
ligand = PDBFile(ligand_pdb)
print(f"Protein: {pdb.topology.getNumAtoms()} atoms, "
      f"{pdb.topology.getNumResidues()} residues")
print(f"Ligand : {ligand.topology.getNumAtoms()} atoms")


Protein: 4391 atoms, 284 residues
Ligand : 23 atoms


## Define force field


In [12]:
# Amber for the protein and the water, the generated template for the ligand. A force
# field is just a list of parameter files, and the ligand's is one more entry.
forcefield = ForceField("amber14-all.xml", "amber14/tip3p.xml", ligand_xml)


## Solvate

In [13]:
# Modeller assembles the system: protein first, then the ligand, then the water.
modeller = Modeller(pdb.topology, pdb.positions)
modeller.add(ligand.topology, ligand.positions)
print(f"Complex: {modeller.topology.getNumAtoms()} atoms "
      f"(protein + {LIGAND}), before solvation")

# Solvate the complex in a box of water
modeller.addSolvent(forcefield, padding=1.0 * nanometer)
print(f"Solvated: {modeller.topology.getNumAtoms()} atoms")


Complex: 4414 atoms (protein + R1P), before solvation
Solvated: 43364 atoms


## setup system

In [14]:
# Create a system. Here we define some forcefield settings such as the nonbonded method.
#
# `hydrogenMass` is what makes the 4 fs timestep below safe. Constraining bonds to
# hydrogen removes their stretching, but hydrogens still move fastest of anything in the
# system, and 4 fs is too long for them. Moving mass from each heavy atom onto the
# hydrogens attached to it slows those motions down without changing the total mass or
# the equilibrium thermodynamics. Without it this system runs for a few thousand steps
# and then dies with "Particle coordinate is NaN", sometimes.
system = forcefield.createSystem(
    modeller.topology,
    nonbondedMethod=PME,
    nonbondedCutoff=1.0 * nanometer,
    constraints=HBonds,
    hydrogenMass=1.5 * amu,
)

# Define the integrator. The Langevin integrator is also a thermostat
integrator = LangevinMiddleIntegrator(300 * kelvin, 1 / picosecond, 0.004 * picoseconds)

# Create the Simulation
simulation = Simulation(modeller.topology, system, integrator)
simulation.context.setPositions(modeller.positions)


In [ ]:
print("Minimizing energy")
simulation.minimizeEnergy()

Minimizing energy


In [ ]:
# helpers
def attach_reporters(simulation: Simulation, dcd: str = "traj.dcd",
                     log: str = "md_log.txt", every: int = 1000, log_every: int = 100,
                     checkpoint: str | None = None,
                     append: bool = False) -> Simulation:
    """Trajectory, screen and log-file reporting for one Simulation object.

    `append=True` extends an existing trajectory and log instead of overwriting them,
    which is what a resumed run needs. Pass `checkpoint` to also write restart files.
    """
    simulation.reporters.append(DCDReporter(dcd, every, append=append))
    simulation.reporters.append(StateDataReporter(
        stdout, every, step=True, potentialEnergy=True, temperature=True, volume=True))
    simulation.reporters.append(StateDataReporter(
        log, log_every, step=True, potentialEnergy=True, temperature=True, volume=True,
        append=append))
    if checkpoint:
        simulation.reporters.append(CheckpointReporter(checkpoint, every))
    return simulation


In [ ]:
attach_reporters(simulation=simulation)


## 4. Analyse the trajectory

Produce and interpret: backbone **RMSD** (has it plateaued?), **potential energy** on a shared
time axis with it (stacked panels), **radius of gyration**, and per-residue **RMSF**.

**Then answer:** how did your section 2 prediction hold up? Name what you got right, what you
got wrong, and what you think you missed.

---

## 5. What the ligand changed

### 5a. The binding site

From the crystal: which residues lie within 4 Å of the ligand, and which contacts are hydrogen
bonds, hydrophobic or electrostatic? Is the ligand **covalently** bound (a `LINK` record naming
it)? If so, say so and stop here. Do the contacts overlap **ex01's conserved positions**?

> **Contact residues:**
>
> **Covalent or non-covalent, and how I checked:**

#### Did your ligand stay where the crystal put it?

Produce and interpret your **ligand's RMSD** after superposing the protein, and its contacts in the
first and last frames: which did it keep, lose and gain? Where do the gripping residues sit on your
RMSF scale? What does that tell you about the pose, the parameters and the protonation, and what not?

> **Kept, lost and gained:**
>
> **Did it stay, and what that tells me:**
>
> **Gripping residues' RMSF against section 2:**

### 5b. Apo against holo

Run the guide's protocol again **without** the ligand (leave out the `modeller.add` line) and
compare the contact residues' RMSF, holo against apo. The guide's own comparison was weak.
Save the holo RMSF first (`rmsf_df.to_csv("rmsf_holo.csv")`), and give the apo run its own
file names (topology, system, trajectory, log and checkpoint), or it overwrites the holo run's.

**Predict before you compare:** will the ligand damp the motion of the residues that grip it?

> **My prediction:**
>
> **What I found:**

### 5c. When the recipe does not fit your ligand

- **A single ion**: Amber's own files cover the common ions; say which you used.
- **A charged ligand**: the script protonates it for pH 7 from typical pKa values. If its
  place in the protein makes that doubtful, say so and say what you did.
- **It fails anyway**: document what you tried and the exact error.

## 6. Compare against your other two structures

Report the three pairwise Cα RMSDs between your **crystal** (`PDB_ID`), your **AlphaFold
model** (ex02) and your **last frame**, and your ligand's RMSD against its crystal pose. A Colab session starts
empty: fetch the model again with ex02's `prediction_record` and `download_prediction_files`, and superpose as the guide's
section 6 does, pairing Cα atoms by residue number.

**Then answer:** is the simulated structure closer to the crystal or to the model, and what
would each outcome mean?

### Question: your last frame against your crystal. What does that prove?

After a run this short, is your protein still the crystal's protein, and is your ligand still where
the crystal put it? Whether your run stayed close or drifted, what does that prove, and what would
you need to tell which reading is right?

> **Still the crystal's protein, and the ligand still in place:**
>
> **What that proves, and what I would need to know more:**

### Question: do termini move more than a folded core in general?

Set your section 4 RMSF beside your ex02 pLDDT: do your termini move more than your folded core, as
2JAC's do? Does "termini are more mobile than a folded core" hold up as a general principle across
your protein, 2JAC and 3D08, or could it be a coincidence? What would you need to check to tell the
difference?

> **On my protein:**
>
> **General principle or coincidence, and what I would check:**

---

## Before you submit

- [ ] **Section 1**, on your ex01 crystal (`PDB_ID`): what was wrong, the parametrisation output, the **net charge**, whether you agree
- [ ] **Section 2:** predictions **written before** the run, citing ex01 and ex02, saying so if they disagree
- [ ] **Section 3:** solvent, force field, integrator, time step, temperature and length **justified**; what you would do with more time
- [ ] **Section 4:** RMSD **plateaued** or flagged as not equilibrated, potential energy beside it, radius of gyration and RMSF, each **interpreted**
- [ ] **Section 4:** section 2 prediction **checked against the result**: right, wrong, missed
- [ ] **Section 5a:** contacts and their types, covalent or not and how you checked, overlap with ex01's conserved positions
- [ ] **Section 5a:** did your ligand stay: RMSD, contacts kept, lost, gained, gripping residues' RMSF, what that tells you
- [ ] **Section 5b:** prediction **written before** comparing, then holo against apo, **or** why it could not be done
- [ ] **Section 5c**, if the recipe did not fit: what you did, or the exact error
- [ ] **Section 6:** three Cα RMSDs, ligand against crystal pose; closer to crystal or model, and what each would mean
- [ ] **Section 6:** what your last frame proves; termini against core in general, and your check
- [ ] Anything that didn't work is **described honestly** rather than deleted: a documented dead end earns marks, a silently missing section does not